# US Large-Cap Factor Screening: Momentum + Quality

**What this does:** scores a basket of 25 US large-cap stocks on two classic
quant factors — **momentum** and **quality** (proxied by low volatility) —
and produces a ranked list. Everything runs end-to-end from the cached data
file, so anyone can re-run it and get the identical ranking.

**Data:** daily closing prices from Yahoo Finance (free, public), cached in
`data/raw_prices.csv` covering 2025-10-01 → 2026-10-16 (263 trading days).
Re-running the notebook uses this cached file, which is what makes the
results reproducible. To refresh with newer data, re-run the download step
described in the README.

In [1]:
import pandas as pd
import numpy as np

TICKERS = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META', 'GOOGL', 'TSLA', 'BRK-B', 'JPM', 'V', 'MA', 'XOM', 'UNH', 'JNJ', 'WMT', 'PG', 'ORCL', 'COST', 'NFLX', 'AMD', 'CRM', 'ABBV', 'LLY', 'AVGO', 'DIS']

prices = pd.read_csv("data/raw_prices.csv", index_col="date", parse_dates=True)
print("Shape:", prices.shape)
print("Range:", prices.index.min().date(), "->", prices.index.max().date())
print("Missing values:", int(prices.isna().sum().sum()))
prices.tail(3)

Shape: (263, 25)
Range: 2025-10-01 -> 2026-10-16
Missing values: 0


,AAPL,MSFT,NVDA,AMZN,META,GOOGL,TSLA,BRK-B,JPM,V,...,PG,ORCL,COST,NFLX,AMD,CRM,ABBV,LLY,AVGO,DIS
date,,,,,,,,,,,,,,,,,,,,,
2026-10-14,342.959991,533.690002,221.300003,254.080002,737.539978,347.640015,367.160004,526.349976,335.279999,391.429993,...,156.529999,130.550003,982.000000,70.570000,570.820007,235.949997,283.480011,1208.739990,346.250000,108.989998
2026-10-15,342.000000,539.440002,217.229996,254.759995,736.719971,349.220001,362.440002,525.679993,335.529999,399.750000,...,157.080002,134.470001,978.010010,72.610001,562.229980,246.740005,283.399994,1251.609985,344.329987,110.680000
2026-10-16,343.459991,529.049988,214.130005,250.000000,709.330017,340.589996,354.720001,532.020020,339.720001,402.609985,...,159.350006,128.229996,989.109985,70.529999,528.750000,240.699997,285.420013,1257.030029,328.750000,110.589996


## Methodology

**Factor 1 — Momentum (6-month trailing return).**
For each stock: `Close(last) / Close(last − 126 trading days) − 1`.
Stocks with stronger recent price trends score higher. This is the standard
academic momentum factor, shortened to a 6-month window to fit one year of data.

**Factor 2 — Quality, proxied by low volatility.**
True quality factors (ROE, accruals, margins) need fundamental data, which is
not freely available in clean form. As a transparent proxy I use the inverse
of 60-day annualized realized volatility:
`1 / (std(daily log returns, 60d) × √252)`.
Calmer price action scores higher — a common stand-in for "quality" in
price-only factor models.

**Composite score.** Each factor is z-scored across the 25 stocks
(`z = (x − mean) / std`), then averaged with equal weights (50/50).
Stocks are ranked by composite score, highest first.

In [2]:
logret = np.log(prices / prices.shift(1))

# Factor 1: 6-month (~126 trading days) trailing return
momentum = prices.iloc[-1] / prices.iloc[-127] - 1

# Factor 2: inverse of 60-day annualized realized volatility (quality proxy)
vol60 = logret.iloc[-60:].std() * np.sqrt(252)
quality_proxy = 1.0 / vol60

factors = pd.DataFrame({"momentum_6m": momentum, "quality_lowvol": quality_proxy})
factors["z_momentum"] = (factors["momentum_6m"] - factors["momentum_6m"].mean()) / factors["momentum_6m"].std()
factors["z_quality"] = (factors["quality_lowvol"] - factors["quality_lowvol"].mean()) / factors["quality_lowvol"].std()
factors["composite"] = 0.5 * factors["z_momentum"] + 0.5 * factors["z_quality"]

ranking = factors.sort_values("composite", ascending=False).reset_index()
ranking.index = ranking.index + 1
ranking.rename(columns={"index": "ticker"}, inplace=True)
cols = ["ticker", "composite", "z_momentum", "z_quality", "momentum_6m", "quality_lowvol"]
print(ranking[cols].round(3).to_string())

   ticker  composite  z_momentum  z_quality  momentum_6m  quality_lowvol
1   BRK-B      1.151       0.009      2.293        0.121           7.218
2     AMD      0.878       3.210     -1.455        0.899           1.588
3       V      0.831       0.621      1.041        0.270           5.337
4    ABBV      0.798       1.032      0.565        0.370           4.622
5      PG      0.724      -0.141      1.590        0.085           6.163
6     JPM      0.639      -0.099      1.378        0.095           5.843
7      MA      0.615       0.269      0.962        0.184           5.218
8     UNH      0.438       0.484      0.392        0.237           4.362
9    AAPL      0.346       0.626      0.067        0.271           3.873
10    LLY      0.344       0.975     -0.287        0.356           3.342
11    XOM      0.344       0.425      0.263        0.222           4.168
12   COST      0.313      -0.534      1.160       -0.011           5.516
13    JNJ      0.140      -0.114      0.394        

In [3]:
# Save the ranking so results.md and re-runs stay in sync
ranking[cols].round(4).to_csv("data/ranking.csv", index_label="rank")
print("Saved data/ranking.csv with", len(ranking), "rows")

Saved data/ranking.csv with 25 rows


## Reading the results

- **Top of the list** = strongest 6-month momentum *and* calmest recent
  price action — the combination this screen rewards.
- **Bottom of the list** = weak momentum and/or very choppy trading.

## Limitations (important)

1. **Price-only model.** Without fundamental data, "quality" is only a
   volatility proxy — it says nothing about earnings, balance sheets, or
   business quality.
2. **One snapshot in time.** Factor leadership rotates; a ranking from
   October 2026 says nothing about next quarter.
3. **Not investment advice.** This is an educational exercise in data
   analysis, not a recommendation to buy or sell anything.